<a href="https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ArishaRamzan-dev/arisha-flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one pseudonymized content item's performance on one report date, for one client, in fact_content_daily_performance, for the month 2026-03 (a mid-panel month, not the sealed final month).


In [ ]:
import duckdb; from google.colab import userdata; HF_TOKEN = userdata.get('HF_TOKEN'); con = duckdb.connect(); con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')"); BASE = "hf://datasets/FlyRank/internship-warehouse"; MONTH = "2026-03"

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature: total_clicks_this_month, total_impressions_this_month, avg_position_this_month, days_with_data, avg_ctr_this_month. Label: whether next month's clicks grew vs this month. Context: content_hash_id, client_hash_id, gsc_data_available. Excluded: query-level data from fact_content_query_90d (not needed at this content-item level). Leak test: adding next month's real clicks as a feature pushed the score from honest 0.596 to a suspicious 1.000 — removed it, kept 0.596.


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
grain_check = con.sql(f"""SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') GROUP BY client_hash_id, content_hash_id, report_date HAVING COUNT(*) > 1 LIMIT 10""").df()
print("Rows violating uniqueness:", len(grain_check))
span_check = con.sql(f"""SELECT COUNT(*) AS total_rows, COUNT(DISTINCT content_hash_id) AS distinct_content, COUNT(DISTINCT client_hash_id) AS distinct_clients, MIN(report_date) AS earliest, MAX(report_date) AS latest FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')""").df()
print(span_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating uniqueness: 0
   total_rows  distinct_content  distinct_clients   earliest     latest
0     9841378            331437                55 2026-03-01 2026-03-31


In [ ]:
availability_check = con.sql(f"""SELECT COUNT(*) AS total_rows, COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')""").df()
print(availability_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378         3611061


In [ ]:
features_df = con.sql(f"""SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS total_clicks_this_month, SUM(gsc_impressions) AS total_impressions_this_month, AVG(gsc_avg_position) AS avg_position_this_month, COUNT(DISTINCT report_date) AS days_with_data FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') WHERE gsc_data_available IS TRUE GROUP BY content_hash_id, client_hash_id""").df()
print(features_df.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id           client_hash_id  total_clicks_this_month  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea                      0.0   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea                      7.0   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea                      0.0   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea                      0.0   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea                      6.0   

   total_impressions_this_month  avg_position_this_month  days_with_data  
0                          57.0                 2.714744              26  
1                        6523.0                 7.209549              31  
2                         149.0                 6.481453              30  
3                         453.0                 2.987198              31  
4                        5630.0                 6.724039              31  


In [ ]:
features_df['avg_ctr_this_month'] = features_df['total_clicks_this_month'] / features_df['total_impressions_this_month'].replace(0, 1)
print(features_df.head())

            content_hash_id           client_hash_id  total_clicks_this_month  \
0  content_05597932fe4da067  client_73cda7b4e4f265ea                      0.0   
1  content_7a105f548d9c6916  client_73cda7b4e4f265ea                      7.0   
2  content_905aa32a0230694e  client_73cda7b4e4f265ea                      0.0   
3  content_a3ea9792f793ec72  client_73cda7b4e4f265ea                      0.0   
4  content_36c36abc7650d7af  client_73cda7b4e4f265ea                      6.0   

   total_impressions_this_month  avg_position_this_month  days_with_data  \
0                          57.0                 2.714744              26   
1                        6523.0                 7.209549              31   
2                         149.0                 6.481453              30   
3                         453.0                 2.987198              31   
4                        5630.0                 6.724039              31   

   avg_ctr_this_month  
0            0.000000  
1       

In [ ]:
next_month_df = con.sql(f"""SELECT content_hash_id, client_hash_id, SUM(gsc_clicks) AS next_month_clicks FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet') GROUP BY content_hash_id, client_hash_id""").df()

leaky_df = features_df.merge(next_month_df, on=['content_hash_id', 'client_hash_id'], how='left')
leaky_df['label'] = (leaky_df['next_month_clicks'] > leaky_df['total_clicks_this_month']).astype(int)

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

honest_features = ['total_clicks_this_month', 'avg_ctr_this_month', 'avg_position_this_month']
X_honest = leaky_df[honest_features].fillna(0)
y = leaky_df['label']
model_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
score_honest = roc_auc_score(y, model_honest.predict_proba(X_honest)[:, 1])
print(f"Honest score: {score_honest:.3f}")

leaky_df['label_leak_column'] = leaky_df['next_month_clicks']
X_leaky = leaky_df[honest_features + ['label_leak_column']].fillna(0)
model_leaky = LogisticRegression(max_iter=1000).fit(X_leaky, y)
score_leaky = roc_auc_score(y, model_leaky.predict_proba(X_leaky)[:, 1])
print(f"Leaky score: {score_leaky:.3f}")

Honest score: 0.596
Leaky score: 1.000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
grain_check = con.sql(f"""SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS n FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet') GROUP BY client_hash_id, content_hash_id, report_date HAVING COUNT(*) > 1 LIMIT 10""").df()
print("Rows violating uniqueness:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows violating uniqueness: 0


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This slice covers only 55 clients active in March 2026; newer clients have no history here. Only 37% of rows have gsc_data_available = TRUE, so results may not generalize to clients without search console access.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.